# 회의록 만들기

녹음 또는 녹취록 → 회의록 → 드라이브/노션. **위에서 아래로 순서대로** 실행합니다.

| # | 셀 | 하는 일 | 만드는 것 |
|---|---|---|---|
| 1 | 설정 | 입력 파일·제목·날짜 지정 | `AUDIO`/`TRANSCRIPT`, `CFG` |
| 2 | STT | 오디오→텍스트 (녹취록이면 건너뜀) | `transcript_path` |
| 3 | 녹취록 확인 | 어느 파일인지·내용 멀쩡한지 확인 | `transcript` |
| 4 | 추출 | `claude -p` 로 회의록 구조 추출 + 검증 | `minutes`, `bundle` |
| 5 | 검토 | 번호 붙여 보여주고 **멈춤** | — |
| 6 | 확정 | 고른 것만 md/html/json 생성 | `out` |
| 7 | 드라이브 | 점검 후 전송 | — |
| 8 | 노션 | API 토큰으로 페이지 생성 | — |

**API 키는 쓰지 않습니다.** 추출은 `claude -p`(Claude Code CLI)가 담당합니다.

**이 노트북은 «들여다보기» 용입니다.** 그냥 회의록만 만들려면 더 쉬운 길이 있습니다.

| | 무엇 | 언제 |
|---|---|---|
| `회의록_GUI.bat` **더블클릭** | GUI. 사람검수 / 오토 모드 | 평소 |
| **이 노트북** | 셀마다 중간 결과를 확인 | 결과가 이상할 때·프롬프트를 손볼 때 |
| `python -m src.pipeline --auto` | 무인 실행 | 스케줄러 |

셋 다 `src/` 의 **같은 함수**를 부릅니다. 노트북에서 잘 된 것은 GUI·CLI 에서도 같게 됩니다.

> ⚠ 이 노트북의 **출력에는 실제 회의 내용이 남습니다.** 커밋 전에
> `Kernel → Restart & Clear All Outputs` 를 실행하세요.

> 커널을 재시작하면 변수가 사라집니다. `out` 이 없다는 오류가 나면 6번까지 다시 실행하세요.
> 4번(추출)은 `data/minutes/draft/*.cli.json` 이 있으면 다시 돌리지 않아도 됩니다.

## 1. 설정

**하는 일** — `.env` 를 읽어 이번 실행에 쓸 값을 정합니다.

### 설정은 `.env` 한 곳에 있습니다

입력 파일·제목·날짜·STT 모델·저장 경로·전송 방식까지 **전부 [`.env`](.env) 에서** 바꿉니다.
각 항목에 설명 주석이 달려 있습니다. 없으면 `.env.example` 을 복사해서 만드세요.

```
1 입력     INPUT_AUDIO / INPUT_TRANSCRIPT / MEETING_TITLE / MEETING_DATE
2 STT      WHISPER_MODEL / WHISPER_DEVICE / STT_LANGUAGE
3 추출     EXTRACT_MODE (cli|api) / ANTHROPIC_API_KEY
4 검토     ACCEPT
5 산출물   OUTPUT_DIR / OUTPUT_LAYOUT / OUTPUT_OVERWRITE
6 전송     SEND (api|sync|manual) / DRIVE_* / SYNC_DIR
7 노션     NOTION_TARGET
```

> **`.env` 를 고쳤으면 커널을 재시작**해야 반영됩니다 (`load_dotenv` 는 import 때 1회만 실행).

### 이 셀에서 임시로 덮어쓰기

아래 `OVERRIDE` 는 «이번 한 번만» 다르게 하고 싶을 때 씁니다. 비워두면 `.env` 값을 씁니다.
상시 설정은 `.env` 에 쓰세요 — 노트북에 적으면 다음에 또 고쳐야 합니다.

In [1]:
# ─── 이번 실행만 다르게 하고 싶을 때. 비우면 .env 값 사용 ───
OVERRIDE = {
    # 'input_transcript': r'data/transcripts/xxx.txt',
    # 'input_audio':      r'data/audio/xxx.m4a',
    # 'meeting_title':    '킥오프',
    # 'meeting_date':     '2026-08-26',
}

# ─── 환경 준비 (여기부터는 건드릴 것 없음) ───
import importlib, os, sys
from pathlib import Path

os.environ.setdefault('PYTHONIOENCODING', 'utf-8')
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    raise SystemExit(f'meeting_minutes 폴더에서 열어야 합니다. 현재: {ROOT}')
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

#  config.py 는 autoreload 대상에서 «제외» 한다.
#  Config 가 dataclass 라서 필드가 바뀌면 autoreload 가 기존 클래스를 갈아끼우지
#  못하고 매 셀 실행마다 아래 오류를 뿜는다:
#     ValueError: __init__() requires a code object with 8 free vars
#  제외해 두면 오류가 사라지고, config.py 를 고쳤을 때만 커널을 재시작하면 된다.
%aimport -src.config

from src.config import CFG, reload as _reload_env

#  .env 를 «매번» 다시 읽는다. 값만 바꿨으면 이 셀 재실행으로 반영된다.
#  (config.py 자체를 고쳤을 때만 커널 재시작이 필요하다)
CFG = _reload_env()

#  config.py 는 dataclass 다. 필드가 추가·삭제되면 %autoreload 가 기존 클래스를
#  갈아끼우지 못하고 «옛 CFG» 가 메모리에 남는다 (ValueError: ... free vars).
#  그 상태로 진행하면 새 속성이 없어 한참 뒤 다른 셀에서 터진다.
#  그래서 «파일에 정의된 필드» 와 «메모리의 CFG» 를 직접 비교한다.
#  목록을 손으로 적으면 필드를 추가할 때마다 여기도 고쳐야 해서 결국 새 필드를 놓친다.
import re as _re

_cfg_src = (ROOT / 'src' / 'config.py').read_text(encoding='utf-8')
import keyword as _kw
_declared = {f for f in _re.findall(r'^\s{4}([a-z_][a-z0-9_]*)\s*:', _cfg_src, _re.M)
             if not _kw.iskeyword(f)}   # try/except 등 키워드 제외
_stale = sorted(f for f in _declared if not hasattr(CFG, f))
if _stale:
    raise SystemExit(
        'CFG 가 낡았습니다. config.py 에 있는데 메모리에 없는 필드: '
        + ', '.join(_stale) + chr(10) +
        '-> Jupyter 상단 «Restart Kernel» 후 1번 셀부터 다시 실행하세요.' + chr(10) +
        '   (.env 값만 바꿨을 때는 이 셀 재실행으로 충분합니다)'
    )

#  커널이 어느 파이썬인지 남긴다. 다른 프로젝트 venv 로 돌면 패키지가 갑자기 없을 수 있다.
print(f'kernel   {sys.executable}')
if 'meeting_minutes' not in sys.executable:
    print('         (주의: 이 프로젝트의 .venv 가 아닙니다. 패키지 누락 시 여기부터 의심)')

for k, v in OVERRIDE.items():
    if v:
        setattr(CFG, k, v)
        print(f'[override] {k} = {v}')
CFG.ensure_dirs()

#  뒤 셀들이 쓰는 값. .env 를 정본으로 하고 여기서 이름만 짧게 받는다.
AUDIO      = CFG.input_audio
TRANSCRIPT = CFG.input_transcript
TITLE      = CFG.meeting_title
DATE       = CFG.meeting_date

print(CFG.summary())
print()
if not AUDIO and not TRANSCRIPT:
    print('입력이 비어 있습니다. .env 의 INPUT_AUDIO 또는 INPUT_TRANSCRIPT 를 채우세요.')
elif AUDIO:
    print(f'입력: 오디오 -> STT 를 돌립니다 ({AUDIO})')
else:
    print(f'입력: 녹취록 -> STT 를 건너뜁니다 ({TRANSCRIPT})')

kernel   c:\Users\skswl\Desktop\Github\v02_quiz_builder\.venv\Scripts\python.exe
         (주의: 이 프로젝트의 .venv 가 아닙니다. 패키지 누락 시 여기부터 의심)
입력      audio=-  transcript=data/transcripts/mom_test1.txt
회의      title=(자동)  date=(미지정)
STT       medium / auto / ko
추출      mode=cli  claude-opus-5/max
검토      accept=(노트북에서 직접)
산출물    C:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\data\minutes  layout=nested  overwrite=False
전송      send=api
드라이브   회의록 / subfolder=slug  gdoc=True
          scope=drive.file
노션      mode=api  토큰=설정됨  대상=있음

입력: 녹취록 -> STT 를 건너뜁니다 (data/transcripts/mom_test1.txt)


## 2. STT — 오디오를 텍스트로

**하는 일** — 오디오면 음성 인식을 돌려 녹취록을 만들고, 녹취록이면 그 파일을 그대로 씁니다.
어느 쪽이든 뒤 셀이 쓸 `transcript_path` 와 `source_name` 을 확정합니다.

**녹취록으로 시작하는 경우에도 이 셀은 실행해야 합니다** — 변수가 여기서 할당됩니다.
STT 만 건너뛰는 것이고, 셀을 건너뛰면 다음 셀에서 `NameError` 가 납니다.

**오디오일 때 알아둘 것**

- 이 PC 는 AMD GPU 라 **CPU 로만 돕니다** (faster-whisper 는 NVIDIA CUDA 전용)
- 진행바에 `0.51x` 처럼 배속이 찍힙니다. 1 미만이면 오디오 길이보다 오래 걸립니다
- 느리면 `.env` 의 `WHISPER_MODEL` 을 `medium` → `small` 로 낮추세요
- 같은 이름의 녹취록이 있으면 덮어쓰지 않고 `_v2` 로 저장합니다

In [2]:
#  입력은 .env 의 INPUT_AUDIO / INPUT_TRANSCRIPT 에서 정합니다.
#  둘 다 비어 있으면 여기서 멈춥니다 (예전엔 Path('') -> '.' 가 되어
#   폴더를 읽으려다 PermissionError 가 났습니다).

if not AUDIO and not TRANSCRIPT:
    raise SystemExit(
        '입력이 비어 있습니다.' + chr(10) +
        '  .env 의 INPUT_AUDIO 또는 INPUT_TRANSCRIPT 를 채우고 커널을 재시작하세요.' + chr(10) +
        '  예) INPUT_TRANSCRIPT=data/transcripts/mom_test1.txt'
    )

if AUDIO:
    from src.transcribe import transcribe
    audio_path = Path(AUDIO)
    #  상대경로면 .env 의 오디오 폴더(data/audio) 기준으로 푼다
    if not audio_path.is_absolute() and not audio_path.exists():
        audio_path = CFG.audio_dir / audio_path.name
    if not audio_path.is_file():
        raise SystemExit(f'오디오 파일이 없습니다: {audio_path}')
    transcript_path = transcribe(audio_path)
    source_name = audio_path.name
else:
    transcript_path = Path(TRANSCRIPT)
    if not transcript_path.is_absolute() and not transcript_path.exists():
        transcript_path = CFG.transcript_dir / transcript_path.name
    if not transcript_path.is_file():
        raise SystemExit(
            f'녹취록 파일이 아닙니다: {transcript_path}' + chr(10) +
            '  .env 의 INPUT_TRANSCRIPT 경로를 확인하세요.'
        )
    source_name = transcript_path.name
    print(f'STT 건너뜀 — 기존 녹취록 사용: {transcript_path}')

STT 건너뜀 — 기존 녹취록 사용: data\transcripts\mom_test1.txt


## 3. 녹취록 확인

**하는 일** — 추출 전에 «어느 파일을 읽었는지, 내용이 멀쩡한지» 눈으로 확인합니다.
파일명·글자수·줄수와 앞 15줄, 뒤 5줄을 보여줍니다.

여기가 깨져 있으면(빈 파일·인코딩 오류·엉뚱한 파일) 뒤 단계가 다 어긋나므로 먼저 봅니다.

`TRANSCRIPT_OVERRIDE` 는 **1·2번을 다시 돌리지 않고 이 셀에서만** 다른 파일을 지정할 때 씁니다.
평소엔 비워두세요.

In [3]:
# ---- 이 셀에서 녹취록을 «직접» 지정하고 싶을 때만 채운다 ----
#  평소엔 비워둡니다. 상시 설정은 .env 의 INPUT_TRANSCRIPT 를 쓰세요.
#  경로는 반드시 r"..." (raw string). r 이 없으면 \U 가 유니코드 이스케이프로
#  해석돼 SyntaxError 가 납니다. 상대경로가 가장 안전합니다.
TRANSCRIPT_OVERRIDE = r''        # 예: r'data/transcripts/mom_test1.txt'

if TRANSCRIPT_OVERRIDE:
    transcript_path = Path(TRANSCRIPT_OVERRIDE)
    if not transcript_path.is_absolute() and not transcript_path.exists():
        transcript_path = CFG.transcript_dir / transcript_path.name
    source_name = transcript_path.name
    print(f'[override] 이 셀에서 지정한 녹취록을 사용합니다: {transcript_path}')

if not transcript_path.is_file():
    raise SystemExit(f'녹취록 파일이 아닙니다: {transcript_path}')

transcript = transcript_path.read_text(encoding='utf-8')
lines = transcript.splitlines()
if not transcript.strip():
    raise SystemExit(f'녹취록이 비어 있습니다: {transcript_path}')

print(f'{transcript_path.name}  |  {len(transcript):,}자 / {len(lines):,}줄')
print('-' * 60)
for l in lines[:15]:
    print(l)
print()
print('... (중략) ...')
print()
for l in lines[-5:]:
    print(l)

mom_test1.txt  |  1,545자 / 57줄
------------------------------------------------------------
[00:00:00] 어 아들 누나랑 지금 태양이랑 밥먹고 있어요 저녁
[00:00:11] 엄마집이 안가깝다
[00:00:16] 맛있는거 먹고있어
[00:00:19] 지민아 저 방 좀 닫아라 저 작은방 석준이 방 너는 가고있냐
[00:00:25] 아 그래 날씨 겁나게 더울걸? 다음주는 더 덥단다 더 더워
[00:00:32] 여긴 막 그렇게 덥진 않은데 다음주엔 더 더워 다음주엔 엄청 더워
[00:00:37] 여기 지금 29도야 그래 왜 시원하대 거기는
[00:00:42] 여기는 35도인데 오늘 낮에 35도
[00:00:47] 그게 막 그렇게 덥진않아
[00:00:49] 엄마는 끈적끈적하지
[00:00:54] 다음주에 강주 38도에 섰구나
[00:00:56] 왜 이렇게 차이가 많이 나?
[00:00:58] 몰라
[00:01:05] 오늘도 태양이랑 놀았어?
[00:01:09] 태양이랑 이제 밥먹고 보낼라고

... (중략) ...

[00:03:27] 누나 끝났어?
[00:03:31] 아니 몰라
[00:03:32] 아직 안 끊었네
[00:03:37] 얼른 가
[00:03:42] 굿나잇


## 4. 추출 — 녹취록을 회의록 구조로

**하는 일** — `claude -p` 로 녹취록을 읽혀 **구조화된 JSON** 을 받고, 검증해서 저장합니다.
API 키를 쓰지 않습니다 (Claude Code 구독).

```
녹취록 ──▶ claude -p ──▶ JSON ──▶ 스키마 검증 ──▶ 인용 검증 ──▶ draft 저장
```

| 어디 | 무엇 |
|---|---|
| 로직 | [`src/extract_cli.py`](src/extract_cli.py) |
| **추출 규칙** | [`prompts/extract_system.md`](prompts/extract_system.md) ← 품질은 여기서 조정 |
| 작업 지시 | [`prompts/extract_task.md`](prompts/extract_task.md) |
| 출력 형식 강제 | [`prompts/extract_output.md`](prompts/extract_output.md) |
| 모델·강도 | `.env` 의 `CLI_MODEL` / `CLI_EFFORT` |

**추출 항목** — `topics` · `decisions` · `action_items` · `open_questions` · `unclear_notes`

**셀이 스스로 검사하는 것**

1. **스키마 검증** — 필드·타입
2. **인용 검증** — 모든 `quote` 가 녹취록에 있는지. 다듬어진 것과 아예 없는 것을 구분
3. **제목·날짜 보정** — 모델이 바꿔놨으면 `.env` 값으로 되돌림

실패해도 **멈추지 않습니다.** 원문을 파일로 남기고 원인 후보를 알려줍니다.
형식을 안 지키면 더 강한 지시로 자동 재시도합니다.

`CLI_EFFORT=max` 는 가장 정확하지만 느립니다. 급하면 `high` 로 낮추세요.

In [4]:
#  로직은 src/extract_cli.py 에, 프롬프트는 prompts/ 에 있습니다.
#  이 셀은 호출과 결과 출력만 합니다.
from src.extract_cli import extract

res = extract(
    transcript_path=transcript_path,
    transcript=transcript,
    source_name=source_name,
    title=TITLE,
    date=DATE,
)
print()
print(res.report())

#  뒤 셀들이 쓰는 이름
if res.ok:
    bundle = res.bundle
    minutes = bundle.minutes
    cli_json = res.draft_path

추출 중... (시도 1/2, 녹취록 길이에 따라 몇 분)

인용 검증: 원문 그대로 3 / 다듬어짐 0 / 불일치 0
결정 1 · 액션 2 · 미결 1 · 확인필요 17
저장: C:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\data\minutes\draft\mom_test1.cli.json


<details><summary>참고 — API 키로 추출하는 경우 (지금은 안 씀)</summary>

`.env` 에 `ANTHROPIC_API_KEY` 가 있으면 4번 셀 대신 아래를 쓸 수 있습니다.
긴 회의를 자동으로 분할 추출·병합해주는 것이 장점입니다.

```python
from datetime import datetime
from src.extract import extract
from src.schema import MinutesBundle

minutes = extract(transcript, title=TITLE or None, date=DATE or None)
bundle = MinutesBundle(minutes=minutes, source_audio=source_name,
                      transcript_chars=len(transcript), model=CFG.model,
                      generated_at=datetime.now().strftime('%Y-%m-%d %H:%M'))
```

단 CLI 경로에 있는 **인용 검증 가드가 없습니다.** 쓰려면 그 부분을 옮겨야 합니다.

</details>

## 5. 검토 — 무엇을 반영할지 고르기

**하는 일** — 추출된 항목에 번호를 붙여 보여줍니다. 문서는 아직 만들지 않습니다.

```
D1 D2 …  결정사항      A1 A2 …  액션아이템      Q1 Q2 …  미결 사항
```

**회의에서 나온 말이 전부 결정은 아닙니다.** 어느 것이 결정인지는 참석한 사람만 알기 때문에
여기서 한 번 멈춥니다.

**같이 나오는 경고**

| 표시 | 뜻 | 다음 행동 |
|---|---|---|
| 회의에서 안 정해짐 | 녹취를 다 봤지만 회의에서 안 정함 | **참석자에게 묻는다** |
| 녹취 불확실 | 녹취가 깨져 확인 못 함 | **원본 오디오를 다시 듣는다** |
| 논의 연결 없음 | 논의 내용에서 근거를 되짚을 수 없음 | 논의가 빠졌나 / 잡담인가 확인 |

In [6]:
from src.review import render_review, blank_report

print(render_review(minutes))

r = blank_report(minutes)
if r["not_stated"]:
    print()
    print(f"! 회의에서 담당/마감을 정하지 않은 액션 {r['not_stated']}건 — 참석자에게 확인")
if r["unclear"]:
    print(f"! 녹취가 불확실해 확인 못 한 액션 {r['unclear']}건 — 원본 오디오 재확인")
if r["notes"]:
    print(f"! 녹취 불확실 구간 {r['notes']}건 — 확정 전 확인 필요")

  가족 안부 통화   
  엄마 마사지는 받지 않기로 하고, 대신 그 비용으로 포도를 사 드리는 쪽으로 정리

[결정사항]
  D1. 엄마 마사지는 받지 않기로 함
      논의: 엄마 마사지 여부
      근거: "아니 안 받아도 돼 지금 요즘에 일 안 한 게 그래도 더 낫어 엄마가" ·00:01:56

[액션아이템]
  A1. 마사지 대신 포도를 사서 엄마에게 챙겨 드리기
      논의: 엄마 마사지 여부
      담당 <녹취 불확실 · 오디오 재확인> / 마감 <회의에서 안 정해짐> / medium
  A2. 제철 과일(포도·복숭아)을 사서 씻고 깎아 챙겨 먹기
      논의: 제철 과일 챙겨 먹기
      담당 <녹취 불확실 · 오디오 재확인> / 마감 <회의에서 안 정해짐> / low

[미결 사항]
  Q1. 누나가 알아보던 마사지가 이미 예약된 상태인지, 취소를 전달해야 하는지 확인되지 않음
      논의: 엄마 마사지 여부

[녹취 불확실 — 확정 전 확인 필요]
  · 전 구간에 화자 라벨이 없다 — 통화 양쪽 발언과 옆사람 발언이 한 줄에 섞여 있어 발언 주체를 특정할 수 없다. 액션 담당자를 모두 unclear 로 둔 이유이며, 담당자 확정에는 원본 오디오 재확인이 필요하다
  · participants 의 "엄마"·"아들"은 호칭이다 — 실명은 확인되지 않았다. "석준"은 "석준이 방"(00:00:19)이라는 방 이름으로만 등장해 발언자로 볼 수 없고, "누나"·"태양"·"지민"은 언급만 되고 본인 발언이 확인되지 않는다
  · "난 차라리 그 놈으로 포도 먹고 싶다"(00:02:09) — "그 돈으로"의 오인식으로 추정. A1 의 근거 문장이므로 우선 재확인 대상
  · "엄마 밭에 가서 일해면 엄마는 또 배로우거든"(00:02:06) — "버거우거든"으로 추정되나 확정 불가. D1 의 사유 구간
  · "엄마집이 안가깝다"(00:00:11) — 의미 불확실. 앞뒤 문맥과 이어지지 않는다
  · "다음주에 강주 38도에 섰

## 6. 확정 — 고른 것만 문서로

**하는 일** — `ACCEPT` 에 적은 항목만 남겨 **md · html · json** 세 파일을 만듭니다.

```
ACCEPT = "all"           전부 반영
ACCEPT = "D1,A1,A3"      고른 것만
```

**만드는 변수** — `out` (세 파일 경로). 뒤의 전송 단계가 이걸 씁니다.

| 파일 | 용도 |
|---|---|
| `.md` | 사람이 읽는 회의록 (노션·슬랙 붙여넣기) |
| `.html` | 뷰어 (액션 표 + 근거 인용) |
| `.json` | 구조 데이터 (다음 자동화 입력) |

고른 항목을 바꾸려면 **이 셀만** 다시 실행하면 됩니다. 같은 제목·날짜면 `_v2` 로 저장됩니다.

In [8]:
ACCEPT = CFG.accept or "all"        # .env 의 ACCEPT. 비우면 all

from src.review import parse_accept, apply_selection
from src.render import render

picked, unknown = parse_accept(ACCEPT, minutes)
assert not unknown, f"알 수 없는 라벨: {unknown}"
assert picked, "선택된 항목이 없습니다"

final = bundle.model_copy(update={"minutes": apply_selection(minutes, picked)})
out = render(final)
print(f"반영 {len(picked)}개: {', '.join(picked)}")
print(f"md   {out.md}")
print(f"html {out.html}")
print(f"json {out.json}")

[render] 같은 이름이 이미 있어 가족_안부_통화_v4 로 저장합니다 (덮어쓰지 않음)
[render] C:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\data\minutes\2026-08-27\가족 에 가족_안부_통화_v4.md / 가족_안부_통화_v4.html / 가족_안부_통화_v4.json
반영 4개: D1, A1, A2, Q1
md   C:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\data\minutes\2026-08-27\가족\가족_안부_통화_v4.md
html C:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\data\minutes\2026-08-27\가족\가족_안부_통화_v4.html
json C:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\data\minutes\2026-08-27\가족\가족_안부_통화_v4.json


### 6-1. 결과 미리보기

**하는 일** — 방금 만든 회의록을 노트북 안에서 바로 보여줍니다. 파일을 열지 않아도 확인됩니다.

`OPEN_HTML = True` 로 두면 HTML 뷰어를 **기본 브라우저로** 엽니다.
노트북 안 `IFrame` 은 쓰지 않습니다 — 산출물이 노트북 폴더 밖(`data/minutes/<날짜>/<제목>/`)에 있어
상대경로가 잡히지 않고 빈 프레임이 마크다운 출력을 덮습니다.

In [9]:
OPEN_HTML = CFG.open_html        # .env 의 OPEN_HTML

from IPython.display import Markdown, display

text = out.md.read_text(encoding="utf-8")
print(f'{out.md.name}  |  {len(text):,}자')
print(f'{out.md.parent}')
print('-' * 60)
display(Markdown(text))

if OPEN_HTML:
    import subprocess
    #  노트북 폴더 밖이라 IFrame 상대경로가 안 잡힌다. 브라우저로 연다.
    subprocess.run(['cmd', '/c', 'start', '', str(out.html)])
    print(f'브라우저로 열었습니다: {out.html.name}')

가족_안부_통화_v4.md  |  2,416자
C:\Users\skswl\Desktop\Github\AI_hub\office_automation\meeting_minutes\data\minutes\2026-08-27\가족
------------------------------------------------------------


# 가족 안부 통화

 · **참석** 엄마, 아들

> 엄마 마사지는 받지 않기로 하고, 대신 그 비용으로 포도를 사 드리는 쪽으로 정리

## 액션아이템

| 할 일 | 논의 | 담당 | 마감 | 우선도 |
|---|---|---|---|---|
| 마사지 대신 포도를 사서 엄마에게 챙겨 드리기 | 엄마 마사지 여부 | **녹취 불확실 · 오디오 재확인** | **회의에서 안 정해짐** | medium |
| 제철 과일(포도·복숭아)을 사서 씻고 깎아 챙겨 먹기 | 제철 과일 챙겨 먹기 | **녹취 불확실 · 오디오 재확인** | **회의에서 안 정해짐** | low |


> **확정 전 확인 필요 2건** — «회의에서 안 정해짐» 은 참석자에게 묻고,
> «녹취 불확실» 은 원본 오디오를 다시 들어야 합니다. 추측으로 채우지 마세요.

## 결정사항

### 1. 엄마 마사지는 받지 않기로 함

- **논의**: 엄마 마사지 여부
- **왜**: 요즘 밭일을 하지 않아 마사지가 필요 없다고 엄마 본인이 밝혔고, 상대가 "알았어"로 수용
- **검토했으나 미채택**: 누나가 알아보던 마사지 받기
- 근거: "아니 안 받아도 돼 지금 요즘에 일 안 한 게 그래도 더 낫어 엄마가" · 00:01:56

## 미결 사항

- 누나가 알아보던 마사지가 이미 예약된 상태인지, 취소를 전달해야 하는지 확인되지 않음 — 확인: 누나 `엄마 마사지 여부`

## 논의 내용

### 제철 과일 챙겨 먹기 `00:01:24`

포도·복숭아 같은 제철 과일을 사서 씻고 깎아 먹으라는 당부. 건강을 이유로 언급됨.

> **A2** 제철 과일(포도·복숭아)을 사서 씻고 깎아 챙겨 먹기

### 엄마 마사지 여부 `00:01:37`

마사지는 누나가 알아보기로 했다는 보고. 엄마는 요즘 밭일을 하지 않아 받지 않아도 된다며 거절하고, 그 비용으로 차라리 포도를 먹고 싶다고 함. 상대가 "알았어"로 수용.

> **D1** 엄마 마사지는 받지 않기로 함
> **A1** 마사지 대신 포도를 사서 엄마에게 챙겨 드리기
> **Q1** 누나가 알아보던 마사지가 이미 예약된 상태인지, 취소를 전달해야 하는지 확인되지 않음

## 확인 필요 (STT 불확실 구간)

- 전 구간에 화자 라벨이 없다 — 통화 양쪽 발언과 옆사람 발언이 한 줄에 섞여 있어 발언 주체를 특정할 수 없다. 액션 담당자를 모두 unclear 로 둔 이유이며, 담당자 확정에는 원본 오디오 재확인이 필요하다
- participants 의 "엄마"·"아들"은 호칭이다 — 실명은 확인되지 않았다. "석준"은 "석준이 방"(00:00:19)이라는 방 이름으로만 등장해 발언자로 볼 수 없고, "누나"·"태양"·"지민"은 언급만 되고 본인 발언이 확인되지 않는다
- "난 차라리 그 놈으로 포도 먹고 싶다"(00:02:09) — "그 돈으로"의 오인식으로 추정. A1 의 근거 문장이므로 우선 재확인 대상
- "엄마 밭에 가서 일해면 엄마는 또 배로우거든"(00:02:06) — "버거우거든"으로 추정되나 확정 불가. D1 의 사유 구간
- "엄마집이 안가깝다"(00:00:11) — 의미 불확실. 앞뒤 문맥과 이어지지 않는다
- "다음주에 강주 38도에 섰구나"(00:00:54) — 지역명 "강주" 불확실(광주·진주 등), 문장 끝도 깨져 있다
- "여기는 35도인데 오늘 낮에 35도"(00:00:42) — 같은 수치가 반복돼 한쪽이 오인식일 수 있다
- "몸 건강해지게. 기관지에."(00:01:35) — "기관지에"가 문맥과 연결되지 않는다. A2 의 목적 구간
- "그거 갖고 왔냐고? 뭐 먹어?"(00:02:23) — "뭐에다 먹냐고"의 오인식으로 추정. 00:02:18~00:02:33 이 같은 질문의 반복 구간이라 실제 발언 횟수를 알 수 없다
- "집밥에 맛이라고 누나가 열심히 먹고 있대"(00:02:48) — 앞부분 의미 불확실
- "딴스러운 말이 나왔다"(00:03:09) — 의미 불확실
- "아빠가 사준 이거 서클이 제일 시원해 이거 서클라이크"(00:03:22) — 제품명 불확실(서큘레이터류로 추정)
- "지민아 저 방 좀 닫아라"(00:00:19) — 호칭 대상 이름 불확실
- "오면은 집밥 먹도 안해"(00:02:59) — 조사·어미가 깨져 주체가 불확실
- 회의 날짜가 녹취록에 없어 date 를 null 로 두었다 — 확정에는 사용자가 --date 로 지정해야 한다
- 날씨·저녁 메뉴·집밥 이야기는 잡담으로 보아 topics 에서 제외했다 (결정·액션으로 뽑지 않은 잡담은 주제도 만들지 않는 규칙)
- "저 방 좀 닫아라"(00:00:19)는 회의 산출물이 아닌 옆사람에 대한 부탁이므로 액션아이템으로 뽑지 않았다

---

<sub>claude-code(claude-opus-5/max) 자동 생성 · 2026-08-27 14:42 · 원본 mom_test1.txt · 녹취 1545자</sub>


## 7. 구글 드라이브 전송

**하는 일** — 회의록 파일을 드라이브로 보냅니다. **어느 계정 · 어느 폴더** 인지 먼저 확인하고,
확인이 통과했을 때만 전송합니다.

```
7-1  확인   계정 · 저장 경로 · 준비 상태     ← 파일을 옮기지 않는다
7-2  전송   7-1 통과 시에만
```

### 경로는 어디서 바꾸나 — 전부 [`.env`](.env)

| 바꿀 것 | `.env` 키 | 예 |
|---|---|---|
| 전송 방식 | `SEND` | `api` / `sync` / `manual` / (비움) |
| 최상위 폴더 이름 | `DRIVE_FOLDER_NAME` | `회의록` |
| 회의별 하위 폴더 | `DRIVE_SUBFOLDER` | `slug` / `month` / `none` |
| 특정 폴더 안에 넣기 | `DRIVE_PARENT_ID` | 폴더 URL 끝의 ID |
| Docs 변환본 함께 | `DRIVE_AS_GDOC` | `true` / `false` |
| 본인 계정 확인용 | `MY_DRIVE_EMAIL` | `you@gmail.com` |
| 동기화 폴더(sync) | `SYNC_DIR` | `G:\My Drive\회의록` |

> `.env` 를 고쳤으면 **커널 재시작** 후 1번 셀부터 다시 실행해야 반영됩니다.

### 7-1. 확인 (전송 안 함)

«어디에 올라갈지» 를 실제 경로로 보여줍니다. 눈으로 확인하고 7-2 로 넘어가세요.

In [10]:
# ===== 7-1. 확인 (파일을 옮기지 않는다) =====
#  경로 설정은 모두 .env 에 있습니다:
#    SEND / DRIVE_FOLDER_NAME / DRIVE_SUBFOLDER / DRIVE_PARENT_ID / DRIVE_AS_GDOC
#    MY_DRIVE_EMAIL (본인 계정 확인용) / SYNC_DIR (sync 방식)
#  1번 셀 없이 이 셀만 돌릴 때를 위해 CFG 를 확보한다.
try:
    CFG
except NameError:
    import sys
    from pathlib import Path
    sys.path.insert(0, str(Path.cwd()))
    from src.config import CFG
    print('[자체 로드] CFG 를 직접 가져왔습니다 (1번 셀 미실행)')

import json as _json

OK_API = OK_SYNC = False
SEND_TARGET = None

print(f'SEND = {CFG.send!r}   (.env 의 SEND)')
print()

# ── api 방식 ──────────────────────────────────────────────
print('[api] Drive API 업로드')
cred, tok = CFG.drive_credentials, CFG.drive_token
if not cred.exists():
    print(f'  X  credentials.json 없음 -> {cred}')
    print('     Cloud Console 에서 «데스크톱 앱» OAuth 클라이언트를 만들어 저장하세요.')
else:
    try:
        d = _json.loads(cred.read_text(encoding='utf-8'))
        kind = next(iter(d))
        if kind != 'installed':
            print(f'  X  유형 {kind!r} — «데스크톱 앱» 으로 다시 만드세요.')
        else:
            OK_API = True
            print(f'  O  credentials.json 정상 (project={d[kind].get("project_id")})')
    except Exception as e:
        print(f'  X  읽기 실패: {e}')

if OK_API:
    #  실제 업로드될 경로를 미리 조립해 보여준다 (drive.py 와 같은 규칙)
    _sub = CFG.subfolder_for(out.slug) if 'out' in dir() else '<회의별 폴더>'
    _top = '내 드라이브' if not CFG.drive_parent_id else f'(부모 {CFG.drive_parent_id})'
    _path = ' / '.join([_top, CFG.drive_folder_name] + ([_sub] if _sub else []))
    print(f'     올릴 위치 : {_path}')
    print(f'     규칙      : DRIVE_SUBFOLDER={CFG.drive_subfolder}  DRIVE_AS_GDOC={CFG.drive_as_gdoc}')
    print(f'     스코프    : {CFG.drive_scope.rsplit("/", 1)[-1]}')
    if tok.exists():
        #  토큰이 있으면 «실제 계정» 을 물어본다. 여기서 계정이 틀리면 지금 멈춰야 한다.
        try:
            from src.drive import _account_email, _service
            print(f'     인증 계정 : {_account_email(_service())}')
        except Exception as e:
            print(f'     인증 계정 확인 실패: {str(e)[:120]}')
    else:
        print('     인증 계정 : (첫 실행 시 브라우저에서 선택 — 본인 계정으로 로그인)')

# ── sync 방식 ─────────────────────────────────────────────
print()
print('[sync] Drive for desktop 동기화 폴더')
try:
    from src.drive_accounts import confirm_target, list_accounts
    for a in list_accounts():
        print(f'  - {a.label}' + (' *현재활성' if a.is_current else ''))
    if CFG.my_drive_email:
        try:
            SEND_TARGET = confirm_target(CFG.my_drive_email, CFG.sync_dir or None)
            OK_SYNC = True
            print(f'  O  올릴 위치: {SEND_TARGET}')
        except SystemExit as e:
            print(f'  X  {str(e).splitlines()[0]}')
    else:
        print('  -  MY_DRIVE_EMAIL 이 비어 확인 생략 (.env 에서 지정)')
except Exception as e:
    print(f'  X  {e}')

print()
print('쓸 수 있는 방식:', ', '.join(
    (['api'] if OK_API else []) + (['sync'] if OK_SYNC else []) + ['manual', '(비움)']))

SEND = 'api'   (.env 의 SEND)

[api] Drive API 업로드
  O  credentials.json 정상 (project=meeting-minutes-mm1-506707)
     올릴 위치 : 내 드라이브 / 회의록 / 가족_안부_통화_v4
     규칙      : DRIVE_SUBFOLDER=slug  DRIVE_AS_GDOC=True
     스코프    : drive.file
     인증 계정 : ada00004@gmail.com

[sync] Drive for desktop 동기화 폴더
  - okgoro9887@thegrowlabs.io  ->  (마운트 미지정) *현재활성
  - duf3283@gmail.com  ->  H:
  X  로그인된 계정 중에 본인 계정이 없습니다.

쓸 수 있는 방식: api, manual, (비움)


### 7-2. 전송

| `SEND` | 동작 | 필요한 것 |
|---|---|---|
| `"api"` | Drive API 자동 업로드 + 공유 링크 | `credentials.json` |
| `"sync"` | 동기화 폴더로 복사 | Drive 앱에 계정 추가 + `SYNC_DIR` |
| `"manual"` | 탐색기 + drive.google.com 열기 | 없음 |
| `""` | 로컬에만 | 없음 |

6-1 이 «쓸 수 있는 방식» 을 알려줍니다. 준비 안 된 방식을 고르면 그 이유를 출력하고 멈춥니다.

In [11]:
# ===== 7-2. 전송 =====
SEND = CFG.send        # .env 의 SEND ('api'|'sync'|'manual'|'')

if SEND == "api":
    assert OK_API, '7-1 에서 credentials.json 확인이 실패했습니다. 위 셀 출력을 보세요.'
    from src.drive import upload_minutes
    #  403 access_denied 가 나오면 원인은 «테스트 사용자 미등록» 이다.
    #  credentials.json 이 있어도 별개 설정이라 따로 해야 한다.
    #    Console -> Google 인증 플랫폼 -> 대상 -> 테스트 사용자 -> + Add users
    #  6-1 점검으로는 잡을 수 없다 (Google 서버 쪽 설정이라 로컬에서 조회 불가).
    if not CFG.drive_token.exists():
        print('첫 실행입니다 — 브라우저가 열립니다. 본인 계정으로 로그인·승인하세요.')
        print('(«확인되지 않은 앱» 경고가 나오면 고급 -> 계속 진행. 본인이 만든 앱입니다)')
    try:
        res = upload_minutes(out.md, out.html, out.json, subfolder=CFG.subfolder_for(out.slug))
        print(res.report())          # 계정 · 저장 위치 · 폴더 링크 · 파일 링크
        print()
        print('드라이브에서 열기:', res.folder_link)
    except Exception as e:
        msg = str(e)
        print(f'업로드 실패: {msg[:200]}')
        if 'access_denied' in msg or '403' in msg:
            print()
            print('원인: 테스트 사용자로 등록되지 않은 계정입니다.')
            print('  Console -> Google 인증 플랫폼 -> 대상 -> 테스트 사용자 -> + Add users')
            print('  본인 이메일을 추가한 뒤 이 셀을 다시 실행하세요.')
        elif 'invalid_grant' in msg or 'expired' in msg:
            print()
            print('원인: token.json 이 만료/무효입니다. 파일을 지우고 다시 실행하세요.')
            print(f'  {CFG.drive_token}')

elif SEND == "sync":
    assert OK_SYNC, '7-1 계정 확인을 통과하지 못했습니다.'
    from src.sync import sync_files
    print(f'대상: {SEND_TARGET}')
    for s in sync_files([out.md, out.html, out.json], subfolder=CFG.subfolder_for(out.slug)):
        print(f'  {s.dst.name}')
    print('Drive 가 백그라운드로 업로드합니다.')

elif SEND == "manual":
    import subprocess
    print(f'탐색기: {out.md.parent}')
    print('브라우저: drive.google.com  — 파일 3개를 드래그하세요')
    subprocess.run(['explorer', '/select,', str(out.md)])
    subprocess.run(['cmd', '/c', 'start', '', 'https://drive.google.com/drive/my-drive'])

else:
    print('전송 건너뜀 — 로컬에만 저장했습니다.')
    for p in (out.md, out.html, out.json):
        print(f'  {p}')

[drive] 폴더 생성: 가족_안부_통화_v4
[drive] 업로드: 가족_안부_통화_v4.md -> https://drive.google.com/file/d/15CWk7khAFnwb34f9XKO5CPRSTGTMid5n/view?usp=drivesdk
[drive] 업로드: 가족_안부_통화_v4.json -> https://drive.google.com/file/d/1wVIU-uTVZ4vRLhoLKIpcjAkZEfNd7CdO/view?usp=drivesdk
[drive] 업로드: 가족_안부_통화_v4.html -> https://drive.google.com/file/d/1UzW9miG31NqxmMjHQi2Xxq0pBnQVkOys/view?usp=drivesdk
[drive] 업로드: 가족_안부_통화_v4 -> https://docs.google.com/document/d/1bwtF-lDd97w6H8XFfFPRdmugtgou11yobfhUbhxFW5Y/edit?usp=drivesdk
계정   ada00004@gmail.com
위치   내 드라이브 / 회의록 / 가족_안부_통화_v4
폴더   https://drive.google.com/drive/folders/15l4PZL8YJKyxeGFIUDgJ0zUHMkN3LTbK

올린 파일
  가족_안부_통화_v4.md
    https://drive.google.com/file/d/15CWk7khAFnwb34f9XKO5CPRSTGTMid5n/view?usp=drivesdk
  가족_안부_통화_v4.json
    https://drive.google.com/file/d/1wVIU-uTVZ4vRLhoLKIpcjAkZEfNd7CdO/view?usp=drivesdk
  가족_안부_통화_v4.html
    https://drive.google.com/file/d/1UzW9miG31NqxmMjHQi2Xxq0pBnQVkOys/view?usp=drivesdk
  가족_안부_통화_v4
    https://docs.google.

## 8. 노션에 올리기

**하는 일** — 회의록을 노션 페이지로 만듭니다.

### 두 방식 (`.env` 의 `NOTION_MODE`)

| | `api` | `mcp` |
|---|---|---|
| 필요한 것 | 토큰 1개 | 없음 |
| 속도 | 1~2초 | 20~60초 |
| 무인 자동화(새벽 스케줄러) | **가능** | 불가 |
| 방법 | 이 셀이 노션 API 호출 | 채팅창에서 Claude 에게 시킴 |

**스케줄러로 돌릴 거면 `api` 여야 합니다.** MCP 커넥터는 대화형 세션에서만 붙습니다.

### api 방식 준비 — 순서대로 (5분)

```
1  노션 -> 설정 -> 연결 -> 우측 상단 «개발자 포털로 이동»
2  «+ 신규 연결»  (또는 상단 «개인 액세스 토큰» 탭)
       이름: meeting-minutes
       유형: 내부(Internal)          <- 공개 아님
       워크스페이스: 본인 계정
3  기능: 콘텐츠 읽기 · 콘텐츠 삽입 체크
4  비밀(Secrets) -> 내부 통합 시크릿 -> 표시 -> 복사   (ntn_... )
5  .env 의 NOTION_TOKEN= 에 붙여넣기

6  ★ 노션 본문으로 돌아가서
       «회의록 자동업로드» 페이지 -> 우측 상단 ••• -> 연결
       -> meeting-minutes 추가
```

**6번을 빠뜨리면 토큰이 있어도 «페이지를 찾을 수 없다» 가 납니다.**
노션은 integration 을 페이지에 «초대» 하는 구조라, 토큰만으로는 아무 페이지도 못 봅니다.

### 어디서 바꾸나

| 바꿀 것 | `.env` 키 |
|---|---|
| 방식 | `NOTION_MODE` (`api` / `mcp` / 비움) |
| 토큰 | `NOTION_TOKEN` |
| 올릴 상위 페이지 | `NOTION_TARGET` (이름 또는 URL) |

### 8-1. 확인 (올리지 않음)

토큰·페이지 접근·계정을 확인합니다. 통과해야 8-2 가 실행됩니다.

In [12]:
# ===== 8-1. 확인 (올리지 않음) =====
#  NOTION_MODE=api 면 토큰으로 직접 확인한다 — Claude 를 부르지 않는다 (1~2초).
#  NOTION_MODE=mcp 면 확인을 건너뛴다 (채팅창에서 사람이 시키는 방식).
try:
    CFG
except NameError:
    import sys
    from pathlib import Path
    sys.path.insert(0, str(Path.cwd()))
    from src.config import CFG
    print('[자체 로드] CFG 를 직접 가져왔습니다 (1번 셀 미실행)')

NOTION_OK = False
print(f'NOTION_MODE   : {CFG.notion_mode or "(비움 — 건너뜀)"}')
print(f'NOTION_TARGET : {CFG.notion_target or "(비어 있음)"}')
print()

if not CFG.notion_mode or not CFG.notion_target:
    print('노션 단계를 건너뜁니다. (.env 의 NOTION_MODE / NOTION_TARGET 확인)')

elif CFG.notion_mode == 'mcp':
    print('mcp 방식 — 이 셀에서는 확인하지 않습니다.')
    print('회의록을 만든 뒤 채팅창에서 Claude 에게 «노션에 올려줘» 라고 하세요.')
    print('무인 자동화(새벽 스케줄러)가 필요하면 .env 의 NOTION_MODE=api 로 바꾸세요.')

elif not CFG.notion_token:
    print('NOTION_TOKEN 이 비어 있습니다. 위 설명 1~5 를 먼저 하세요.')

else:
    from src.notion import check as notion_check
    info = notion_check()          # 토큰으로 직접 조회. Claude 안 부름
    print('연결 계정  :', info['account'] or '(확인 안 됨)')
    print('올릴 상위  :', info['parent_title'] or '(확인 안 됨)')
    NOTION_OK = info['ok']
    if not NOTION_OK:
        print()
        print(info['error'])

print()
print('노션 업로드 가능:', NOTION_OK)

NOTION_MODE   : api
NOTION_TARGET : https://app.notion.com/p/3c9573b8547b80a5adf5e07f34a4e470

연결 계정  : meeting-minutes
올릴 상위  : 회의록 자동업로드

노션 업로드 가능: True


### 8-2. 올리기

**하는 일** — `Minutes` 구조를 노션 블록으로 직접 만들어 페이지를 생성합니다.
마크다운을 거치지 않아서 **액션은 체크박스, 결정은 인용과 함께** 들어갑니다.

`mcp` 방식이면 이 셀은 «채팅창에 붙여넣을 문구» 만 출력합니다.

In [13]:
# ===== 8-2. 올리기 =====
if CFG.notion_mode == 'api':
    assert NOTION_OK, '8-1 확인을 통과하지 못했습니다. 위 셀 출력을 보세요.'
    from src.notion import upload as notion_upload
    res = notion_upload(bundle)
    print(res.report())

elif CFG.notion_mode == 'mcp':
    #  채팅창에서 Claude 에게 시키는 경로. 파일 «경로» 만 넘긴다 —
    #  회의 내용이 채팅에 그대로 실리지 않게.
    print('아래를 채팅창에 붙여넣으세요:')
    print()
    print(f'{out.md} 이 회의록을 노션 «{CFG.notion_target}» 에 올려줘.')
    print('액션아이템은 체크박스로, 결정사항은 근거 인용과 함께.')
    print('<회의에서 안 정해짐> / <녹취 불확실> 표기는 그대로 유지해줘.')

else:
    print('NOTION_MODE 가 비어 있어 노션 단계를 건너뜁니다.')

계정   meeting-minutes
상위   회의록 자동업로드
생성   https://app.notion.com/p/3c9573b8547b81cbb28ee880601d5a8b
블록   39개


---

## 다시 돌릴 때

| 하고 싶은 것 | 실행할 셀 |
|---|---|
| 고르는 항목만 바꾸기 | 5번만 (`ACCEPT` 수정) |
| 추출 품질이 아쉬움 | `prompts/extract_system.md` 고치고 3번부터 |
| 다른 회의 | 1번부터 (`AUDIO`/`TITLE`/`DATE` 수정) |
| STT 다시 | 2번부터 (`.env` 의 `WHISPER_MODEL` 조정) |

**커밋 전에 `Kernel → Restart & Clear All Outputs`** — 출력에 회의 내용이 남아 있습니다.